In [1]:
import yaml

fp = "datasets.yaml"
with open(fp, "r") as f:
    datasets = yaml.safe_load(f)
list(datasets.keys())

In [22]:
# list(datasets.values())[0].keys() #['Quality_assurance']['qualitative_assessment']
import pandas as pd


def get_data(dataset_name, entry):
    organization = entry["Description_Organization_Dataset"]
    origin = entry["Origin_Content"]
    topics_relevance = entry["Topics/Source Relevancy"]["qualitative_assessment"]
    quality_assurance = entry["Quality_assurance"]["qualitative_assessment"]
    data = {
        "dataset_name": dataset_name,
        "organization": organization,
        "origin": origin,
        "topics_relevance_score": topics_relevance["score"],
        "topics_relevance_explanation": topics_relevance["explanation"],
        "quality_assurance_score": quality_assurance["score"],
        "quality_assurance_explanation": quality_assurance["explanation"],
    }
    return data


all_data = {}
for dataset_name, dataset_entry in datasets.items():
    data = get_data(dataset_name, dataset_entry)
    all_data[dataset_name] = data

quality = pd.DataFrame.from_dict(all_data, orient="index")

# Subsample 6 rows for display, groupby quality_assurance_score. I want two rows per score.
display_df = quality[quality["quality_assurance_score"] != "low/medium/high"]
display_df = (
    display_df.groupby("quality_assurance_score")
    .apply(lambda x: x.sample(n=3, random_state=10))
    .reset_index(drop=True)
)
# Define custom order for sorting
score_order = {"high": 3, "medium": 2, "low": 1}
display_df["sort_key"] = display_df["quality_assurance_score"].map(score_order)
display_df = display_df.sort_values(by="sort_key", ascending=False).drop(
    "sort_key", axis=1
)
print(
    display_df[
        [
            "dataset_name",
            "origin",
            "organization",
            "quality_assurance_score",
            "quality_assurance_explanation",
        ]
    ].to_markdown()
)
# display_df